# Multimer analysis III - scVI intergration
We observed in analysisII that merging the main data and the multimer datasets using the original HVGs from the main dataset clearly showed broad batch effects (main vs multimer datasets). Here we first explore batch effects between the two multimer sorts that we did, and then we intergrate the two multimer sorts with scVI, and export

In [ ]:
import anndata as ad
import scanpy as sc
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import muon as mu
import scvi
import torch
# from scipy.stats import binomtest
# from scipy.stats import mannwhitneyu
# import string

# Helps to define plotting area
import matplotlib.pyplot as plt
# from matplotlib import rcParams 
# from matplotlib.pyplot import rc_context
# import matplotlib.gridspec as gridspec
# import matplotlib.colors as mcolors
# from matplotlib import colors
# # from matplotlib_venn import venn2
# from scipy.stats import spearmanr
# import matplotlib.patches as mpatches
# from pathlib import Path

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

In [ ]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

In [ ]:
torch.set_float32_matmul_precision("high")

# Loading data
## Step 1 - Load all multimer data

In [ ]:
mdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_multimer_dataset_preprocessed_annotated.h5mu")
len(mdata)

#### Preprocess slightly

In [ ]:
## set NA for clones that include "-nan"
mdata["gex"].obs["clone_handle"] = np.where(mdata["gex"].obs["clone_handle"].str.contains("-nan"), pd.NA, mdata["gex"].obs["clone_handle"])
mdata["airr"].obs["clone_handle"] = mdata["gex"].obs["clone_handle"].copy()

## remove the ".0" from all clone_handles in mdata
mdata["gex"].obs["clone_handle"] = (
    mdata["gex"].obs["clone_handle"]
    .str.replace(r"\.0$", "", regex=True)
)

In [ ]:
## Filter to only include multimer positive cells
# mu.pp.filter_obs(mdata["gex"], 'handle', lambda x: (x.notna()))
# mdata.update()

In [ ]:
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = mdata["gex"].obs.index.tolist()

## define a cell index
mdata["airr"].obs["cell_index"] = mdata["airr"].obs.index
mdata["prot"].obs["cell_index"] = mdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
mdata["airr"].obs["filter_index"] = pd.Categorical(np.where(mdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
mdata["prot"].obs["filter_index"] = pd.Categorical(np.where(mdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that mdata["airr"].obs and mdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(mdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(mdata["prot"], 'filter_index', lambda x: x == "GEX_present")

mdata.update()

In [ ]:
len(mdata)

# Batch effects even among two 10x lanes in the multimer sort

In [ ]:
## Reuse HVGs from main analysis
HVG = pd.read_csv("../../06_csv_outputs/HVG/CD8_HVGs_export_after_TCR_filter.csv")
list_of_HVGs = HVG[HVG["highly_variable"] == True]["genes"].tolist()

mdata["gex"].var["highly_variable"] = np.where(mdata["gex"].var.index.isin(list_of_HVGs), True, False)

In [ ]:
sc.tl.pca(mdata["gex"])
sc.pp.neighbors(mdata["gex"])
sc.tl.umap(mdata["gex"])

In [ ]:
sc.pl.umap(mdata["gex"], color="batch", legend_loc="on data", legend_fontweight= 'bold', frameon= False , size=5, vmin=0, legend_fontoutline=2)

# scVI intergration
Another important thing to keep in mind is highly-variable gene selection. While scVI and scANVI both accomodate for large gene sets in terms of runtime, we usually recommend filtering genes for best performance when the dataset has few number of cells. As a rule of thumb, performance starts to decrease when number of cells and number of genes are comparable. This point is emphasized in this comparative analysis of data integration algorithms for scRNA-seq data.
- See: https://docs.scvi-tools.org/en/stable/tutorials/notebooks/scrna/tabula_muris.html

For best practices of how/when to perform feature selection, please refer to the model-specific tutorial. For scVI, we recommend anywhere from 1,000 to 10,000 HVGs, but it will be context-dependent.
- See: https://docs.scvi-tools.org/en/1.3.3/tutorials/notebooks/quick_start/api_overview.html

We recomend that data should be subsetted to HVGs before integration. In our paper we selected HVGs per system (e.g. species) with Scanpy function pp.highly_variable_genes using within-system batches as the batch_key, starting with genes present in all systems. Then, we took the intersection of HVGs across systems to obtain ~2000 shared HVGs. The here-used example data was already preprocessed accordingly and has normalized data in X.
- See: https://docs.scvi-tools.org/en/stable/tutorials/notebooks/scrna/sysVI.html

In [ ]:
# scVI works better with fewer genes - subset to the highly variable genes
## 2000 genes is selected due to the comment in the tabula muris tutorial that the performance starts to decrease, when the number of genes and cells are comparable. Here the number of genes and cells are comparable with 2000, and should not be increased further. 
sc.pp.highly_variable_genes(
    mdata["gex"],
    flavor="seurat_v3",
    n_top_genes=2000,
    layer="counts",
    batch_key="batch",
    subset=True,
)

In [ ]:
# Intergration using scVI
scvi.model.SCVI.setup_anndata(mdata["gex"], layer="counts", batch_key="batch")

In [ ]:
scvi_model = scvi.model.SCVI(mdata["gex"], n_layers=2, n_latent=30)

In [ ]:
scvi_model.train()

In [ ]:
# Add the latent representation to the anndata gex object
SCVI_LATENT_KEY = "X_scVI"
mdata["gex"].obsm[SCVI_LATENT_KEY] = scvi_model.get_latent_representation()

In [ ]:
sc.pp.neighbors(mdata["gex"], use_rep=SCVI_LATENT_KEY)
sc.tl.umap(mdata["gex"], min_dist=0.5)

In [ ]:
sc.pl.umap(mdata["gex"], color="batch", legend_fontweight= 'bold', frameon= False , size=10, vmin=0, legend_fontoutline=2)

In [ ]:
sc.pl.umap(mdata["gex"], color="donor", legend_fontweight= 'bold', frameon= False , size=10, vmin=0, legend_fontoutline=2)

## Annotate each multimer with an antigen
This is a manually made annotation sheet

In [ ]:
Ag_ann = pd.read_excel("../../03_annotation/11_multimer_experiment_ann/2026_03_antigen_annotation_per_sequence.xlsx")

# Create a dictionary where you can map values into a column
Ag_dict = dict(zip(Ag_ann["Sequence"], Ag_ann["Antigen"]))

# Map values into the dataframe as an antigen column
mdata["gex"].obs['Antigen'] = mdata["gex"].obs['Sequence'].apply(lambda x: Ag_dict[str(x)])

## Annotate each cells with sort category

In [ ]:
mdata["gex"].obs["catsort_virus"] = np.where(mdata["gex"].obs["Virus"] == "EBV", "EBV_sort", "Non_EBV_sort")
mdata["gex"].obs["catsort_A2"] = np.where(mdata["gex"].obs["HLA"] == "A0201", "A2_sort", "NonA2_sort")


## Fix the presence of Virus labels among non-expanded cells

In [ ]:
mdata["gex"].obs["handle"] = pd.Categorical(np.where(mdata["gex"].obs["clone_size"] >=2, mdata["gex"].obs["handle"], pd.NA))
mdata["gex"].obs["Virus"] = pd.Categorical(np.where(mdata["gex"].obs["clone_size"] >=2, mdata["gex"].obs["Virus"], pd.NA))
mdata["gex"].obs["Sequence"] = pd.Categorical(np.where(mdata["gex"].obs["clone_size"] >=2, mdata["gex"].obs["Sequence"], pd.NA))
mdata["gex"].obs["HLA"] = pd.Categorical(np.where(mdata["gex"].obs["clone_size"] >=2, mdata["gex"].obs["HLA"], pd.NA))
mdata["gex"].obs["Antigen"] = pd.Categorical(np.where(mdata["gex"].obs["clone_size"] >=2, mdata["gex"].obs["Antigen"], pd.NA))

## Remove unused categories

In [ ]:
mdata["gex"].obs["handle"] = mdata["gex"].obs["handle"].cat.remove_unused_categories()
mdata["gex"].obs["Virus"] = mdata["gex"].obs["Virus"].cat.remove_unused_categories()
mdata["gex"].obs["Sequence"] = mdata["gex"].obs["Sequence"].cat.remove_unused_categories()
mdata["gex"].obs["HLA"] = mdata["gex"].obs["HLA"].cat.remove_unused_categories()

## Annotate whether a cell is antigen-specific: Yes or No

In [ ]:
mdata["gex"].obs["catagspecifc"] = np.where(~mdata["gex"].obs["handle"].isna(), "agspecific", pd.NA)
print("I have this many rows total: ", len(mdata["gex"].obs), ". I have this many rows that are ag-specific: ", len(mdata["gex"].obs[mdata["gex"].obs["catagspecifc"] == "agspecific"]))


## Calculate total_clone_count

In [ ]:
mdata["gex"].obs["total_clone_count"] = mdata["gex"].obs.groupby("clone_handle").transform("size").fillna(0)

## Calculate number of antigen-specific cells sorted for each donor

In [ ]:
mdata["gex"].obs["total_agspecific_cells"] = mdata["gex"].obs.groupby(["catagspecifc", "donor", "catsort_A2", "catsort_virus"], observed=True)["catsort_A2"].transform("size")
mdata["gex"].obs["total_agspecific_cells"] = mdata["gex"].obs["total_agspecific_cells"].fillna(0)

#Check
mdata["gex"].obs.groupby(["donor", "catsort_A2", "catsort_virus", "total_agspecific_cells", "catagspecifc"], observed=True).size()

## Filter out non-specific cells from Pt17/21

In [ ]:
mu.pp.filter_obs(mdata["gex"], 'donor', lambda x: x != "Pt17/Pt21")
mdata["gex"].obs["donor"] = mdata["gex"].obs["donor"].cat.remove_unused_categories()
mdata.update()

## Preprocess - mask responses in CMV-negative individuals to NA

#### Set the CMV-specific responses observed in CMV-negative individuals to NA
These are masked before analysis

In [ ]:
mdata["gex"].obs["CMV_sero_resp_eval"].drop_duplicates()

In [ ]:
mdata["gex"].obs["HLA"]

In [ ]:
## Set seronegative, CMV-specific T cells to NA
mdata["gex"].obs["handle"] = pd.Categorical(np.where(mdata["gex"].obs["CMV_sero_resp_eval"] == "Seronegative, CMV-specific T cells", pd.NA, mdata["gex"].obs["handle"]))
mdata["gex"].obs["Virus"] = pd.Categorical(np.where(mdata["gex"].obs["CMV_sero_resp_eval"] == "Seronegative, CMV-specific T cells", pd.NA, mdata["gex"].obs["Virus"]))
mdata["gex"].obs["HLA"] = pd.Categorical(np.where(mdata["gex"].obs["CMV_sero_resp_eval"] == "Seronegative, CMV-specific T cells", pd.NA, mdata["gex"].obs["HLA"]))
mdata["gex"].obs["Sequence"] = pd.Categorical(np.where(mdata["gex"].obs["CMV_sero_resp_eval"] == "Seronegative, CMV-specific T cells", pd.NA, mdata["gex"].obs["Sequence"]))

In [ ]:
sc.pl.umap(mdata["gex"], color="Virus", legend_fontweight= 'bold', frameon= False , size=20, vmin=0, legend_fontoutline=2)

# Preprocess - mask all FRC responses

In [ ]:
## Set seronegative, CMV-specific T cells to NA
mdata["gex"].obs["handle"] = pd.Categorical(np.where(mdata["gex"].obs["Sequence"] == "FRCPRRFCF", pd.NA, mdata["gex"].obs["handle"]))
mdata["gex"].obs["Virus"] = pd.Categorical(np.where(mdata["gex"].obs["Sequence"] == "FRCPRRFCF", pd.NA, mdata["gex"].obs["Virus"]))
mdata["gex"].obs["HLA"] = pd.Categorical(np.where(mdata["gex"].obs["Sequence"] == "FRCPRRFCF", pd.NA, mdata["gex"].obs["HLA"]))
mdata["gex"].obs["Sequence"] = pd.Categorical(np.where(mdata["gex"].obs["Sequence"] == "FRCPRRFCF", pd.NA, mdata["gex"].obs["Sequence"]))

# Preprocess - calculate the CSF-biased score

In [ ]:
DEG_biased = pd.read_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_filtered_DEGs.csv")["names"].drop_duplicates()
print(len(DEG_biased))

# Cacluate a score
sc.tl.score_genes(mdata["gex"], 
                      DEG_biased,
                      score_name='CSF-biased score')

DEG_biased = pd.read_csv("../../06_csv_outputs/DEG/CD8_PB_biased_filtered_DEGs.csv")["names"].drop_duplicates()
print(len(DEG_biased))
# Cacluate a score
sc.tl.score_genes(mdata["gex"], 
                      DEG_biased,
                      score_name='PB-biased score')


In [ ]:
sc.settings.set_figure_params(figsize=(4.5, 4))
fig = sc.pl.umap(mdata["gex"], color="CSF-biased score", legend_fontweight= 'bold', cmap="RdBu_r", frameon= False , size=40, vmax=0.4, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/5_CSFbiased_score_umap.pdf",  format="pdf", dpi=400)
plt.show

In [ ]:
sc.settings.set_figure_params(figsize=(4.5, 4))
fig = sc.pl.umap(mdata["gex"], color="PB-biased score", legend_fontweight= 'bold', cmap="RdBu_r", frameon= False , size=40, vmax=0.4, legend_fontoutline=2, return_fig=True, show=False)
plt.tight_layout()
fig.savefig("../../05_plots/09_multimers/5_PBbiased_score_umap.pdf",  format="pdf", dpi=400)
plt.show

## Export .csv file with clone_handle and multimer-specificity information

In [ ]:
# Get labels from airr into gex
mdata["gex"].obs["IR_VJ_1_cdr1aa"] = ir.get.airr(mdata, "cdr1_aa", ["VJ_1"])
mdata["gex"].obs["IR_VJ_1_cdr2aa"] = ir.get.airr(mdata, "cdr2_aa", ["VJ_1"])
mdata["gex"].obs["IR_VJ_1_v_call"] = ir.get.airr(mdata, "v_call", ["VJ_1"])
mdata["gex"].obs["IR_VJ_1_j_call"] = ir.get.airr(mdata, "j_call", ["VJ_1"])
mdata["gex"].obs["IR_VJ_1_junction_aa"] = ir.get.airr(mdata, "junction_aa", ["VJ_1"])

mdata["gex"].obs["IR_VJ_2_cdr1aa"] = ir.get.airr(mdata, "cdr1_aa", ["VJ_2"])
mdata["gex"].obs["IR_VJ_2_cdr2aa"] = ir.get.airr(mdata, "cdr2_aa", ["VJ_2"])
mdata["gex"].obs["IR_VJ_2_v_call"] = ir.get.airr(mdata, "v_call", ["VJ_2"])
mdata["gex"].obs["IR_VJ_2_j_call"] = ir.get.airr(mdata, "j_call", ["VJ_2"])
mdata["gex"].obs["IR_VJ_2_junction_aa"] = ir.get.airr(mdata, "junction_aa", ["VJ_2"])

mdata["gex"].obs["IR_VDJ_1_cdr1aa"] = ir.get.airr(mdata, "cdr1_aa", ["VDJ_1"])
mdata["gex"].obs["IR_VDJ_1_cdr2aa"] = ir.get.airr(mdata, "cdr2_aa", ["VDJ_1"])
mdata["gex"].obs["IR_VDJ_1_v_call"] = ir.get.airr(mdata, "v_call", ["VDJ_1"])
mdata["gex"].obs["IR_VDJ_1_d_call"] = ir.get.airr(mdata, "d_call", ["VDJ_1"])
mdata["gex"].obs["IR_VDJ_1_j_call"] = ir.get.airr(mdata, "j_call", ["VDJ_1"])
mdata["gex"].obs["IR_VDJ_1_junction_aa"] = ir.get.airr(mdata, "junction_aa", ["VDJ_1"])

mdata["gex"].obs["IR_VDJ_2_cdr1aa"] = ir.get.airr(mdata, "cdr1_aa", ["VDJ_2"])
mdata["gex"].obs["IR_VDJ_2_cdr2aa"] = ir.get.airr(mdata, "cdr2_aa", ["VDJ_2"])
mdata["gex"].obs["IR_VDJ_2_v_call"] = ir.get.airr(mdata, "v_call", ["VDJ_2"])
mdata["gex"].obs["IR_VDJ_2_d_call"] = ir.get.airr(mdata, "d_call", ["VDJ_2"])
mdata["gex"].obs["IR_VDJ_2_j_call"] = ir.get.airr(mdata, "j_call", ["VDJ_2"])
mdata["gex"].obs["IR_VDJ_2_junction_aa"] = ir.get.airr(mdata, "junction_aa", ["VDJ_2"])


In [ ]:
obs_export = mdata["gex"].obs[["clone_handle", "handle", "Virus", "Antigen", "HLA", "Sequence", 
                               'IR_VJ_1_junction_aa',  'IR_VJ_2_junction_aa',  'IR_VDJ_1_junction_aa',  'IR_VJ_1_cdr1aa',
                               'IR_VJ_1_cdr2aa', 'IR_VJ_1_v_call',
                                'IR_VJ_1_j_call',
                                'IR_VJ_2_cdr1aa',
                                'IR_VJ_2_cdr2aa',
                                'IR_VJ_2_v_call',
                                'IR_VJ_2_j_call',
                                'IR_VDJ_1_cdr1aa',
                                'IR_VDJ_1_cdr2aa',
                                'IR_VDJ_1_v_call',                              
                               'IR_VDJ_1_d_call',
                               'IR_VDJ_1_j_call',]].drop_duplicates()

#Save locally
obs_export.to_excel("../../03_annotation/11_multimer_experiment_ann/clone_x_multimer_assignments.xlsx")

In [ ]:
obs_export.sort_values("handle").head(10)

# Export 

In [ ]:
mdata.write(filename="../../04_data_objects/06_preprocessed_v2/2026_6_multimer_dataset_preprocessed_annotated_intgr.h5mu")